# 5. After the milestone: feature improvements

Run later, after inspecting baseline errors. These are exploratory tests on the same four subject-disjoint folds: time versus time + frequency, back/thigh removal, magnitude-only features, class-balanced RF, LR with/without PCA and 2 versus 4 seconds.

Compare variants within the four-fold table, separately from LOSO. The 4-second variant changes voting targets and sample count. No outcome is assumed to be better.

Training is disabled below. Set RUN_IMPROVEMENTS to True only when ready. Opening this notebook does not train models.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
            if (p / 'har_utils.py').exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

## Define the later experiments here

This orchestration function belongs only to notebook 05. It uses the shared feature extraction, fixed model definitions and evaluation helpers so the comparisons stay consistent. Defining it does not start training.

Follow each call: choose one representation, use the same four subject folds, then compare with the matched control. The 4-second test rebuilds windows explicitly.

In [ ]:
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def run_feature_experiments():
    """Optional notebook 05 experiments, using the same four subject folds."""
    X, meta = load_windows(2)
    forest = models()['RF']
    for representation in ['time', 'time_frequency', 'back', 'thigh', 'magnitude']:
        evaluate(X[feature_columns(X, representation)], meta, forest,
                 'RF_' + representation, 'group4')
    balanced = clone(forest).set_params(class_weight='balanced_subsample')
    evaluate(X[feature_columns(X)], meta, balanced, 'RF_balanced', 'group4')
    pca = make_pipeline(StandardScaler(), PCA(n_components=.95, svd_solver='full'),
                        LogisticRegression(max_iter=1500, random_state=SEED))
    evaluate(X[feature_columns(X)], meta, models()['LR'], 'LR_time', 'group4')
    evaluate(X[feature_columns(X)], meta, pca, 'LR_pca95', 'group4')
    X4, meta4 = build_windows(4)
    evaluate(X4[feature_columns(X4)], meta4, forest, 'RF_4seconds', 'group4')
    return save_summary()

In [ ]:
RUN_IMPROVEMENTS = False
if RUN_IMPROVEMENTS:
    run_feature_experiments()
else:
    print('Feature experiments are deferred. No training started.')

The local experiment function prepares 4-second windows only when its training flag is enabled. Frequency features cached during preparation are not used in milestone model comparisons. Each variant changes one factor relative to its matched control.

In [ ]:
files = sorted((ROOT / 'results').glob('*_group4_folds.csv'))
if files:
    scores = pd.concat([pd.read_csv(p) for p in files])
    table = summarize(scores)
    display(table.round(4))
    table.set_index('experiment').macro_f1.sort_values().plot.barh(figsize=(10, 5))
    plt.xlabel('Mean 12-class macro F1'); plt.tight_layout(); plt.show()
else:
    print('No improvement results yet. Run the optional training cell when ready.')

## Extensions to implement after reviewing the errors

Gravity/movement separation and axis correlations are follow-up candidates, not implemented or claimed results. Add one at a time and compare against the same time-feature control. Preserve continuity and document filtering at recording boundaries.

## Interpretation

Check per-class recall and fold variation before choosing a feature set. Keep negative results. Four-fold exploration does not establish unbiased final performance after selecting the best variant. Feature selection must also occur inside outer training folds if you want to evaluate the complete selection procedure.